In [ ]:
import torch

# testing if multihead concat is equivalent to sum of heads 

d = 64
h = 4
d_out = 32
T = 10

X_h = torch.randn(h, T, int(d/h))
X = X_h.permute(1, 0, 2).contiguous().view(T, d)

W_o_h = torch.randn(h, int(d/h), d_out)

W_o = torch.cat([W_o_h[i] for i in range(h)], dim=0)

Y_concat = X @ W_o
Y_sum = torch.stack([X_h[i] @ W_o_h[i] for i in range(h)]).sum(dim=0)

# check if Y_concat and Y_sum are close
print(torch.norm(Y_concat - Y_sum).item())


2.677010343177244e-05


In [47]:
import numpy as np

m = 2 # modalities
n = 2 # experts
p = 2 # tasks

# routing matrix of shape (m, n, p)
X = np.random.rand(m, n, p) # (modality, expert, task)

# given an input, p(expert j | task i)
def p_ej_tk(X: np.ndarray, task_idx: int, modality_idx: int, expert_idx: int) -> np.ndarray:
    nominator = np.sum(X[:, expert_idx, task_idx])
    denominator = np.sum(X[modality_idx, :, :])
    return nominator / denominator

def p_mi_tk(X: np.ndarray, task_idx: int, modality_idx: int, expert_idx: int) -> np.ndarray:
    nominator = np.sum(X[modality_idx, :, task_idx])
    denominator = np.sum(X[:, expert_idx, :])
    return nominator / denominator

def p_mi_ej_tk(X: np.ndarray, task_idx: int, modality_idx: int, expert_idx: int) -> np.ndarray:
    nominator = np.sum(X[modality_idx, :, task_idx])
    denominator = np.sum(X[:, :, :])
    return nominator / denominator

# testing
task_idx = 0
modality_idx = 0
expert_idx = 0
print(f"p(expert_{expert_idx} | task_{task_idx}) = {p_ej_tk(X, task_idx, modality_idx, expert_idx)}")
print(f"p(modality_{modality_idx} | task_{task_idx}) = {p_mi_tk(X, task_idx, modality_idx, expert_idx)}")
print(f"p(modality_{modality_idx} , expert_{expert_idx} | task_{task_idx}) = {p_mi_ej_tk(X, task_idx, modality_idx, expert_idx)}")

p(expert_0 | task_0) = 0.7842232414282003
p(modality_0 | task_0) = 0.4634647952475315
p(modality_0 , expert_0 | task_0) = 0.23792673082912513


In [ ]:
# testing scenario: 2 experts, 2 modalities, 2 tasks
# given an input of task 0, modality 0, what is the probability of expert 0 being selected for task 0?

In [ ]:
task_idx = 0
modality_idx = 0

joint_sum = sum(
    p_ej_tk(X, task_idx, modality_idx, j) for j in range(n)
)
print(f"Sum of P(E_j | T_0) over j: {joint_sum:.4f}")

Sum of P(E_j | T_0) over j: 1.3342


In [ ]:
# nominator: how much expert j is used for task k, across all modalities
# denominator: how much modality i contributes in total, across all experts and tasks
# the ratio says out of everything modality i contributes, how much of it goes to expert j for task k